# Dynamics and Numerical Integration

## Table of Contents

1. [The Simple Pendulum](#the-simple-pendulum)
   - [Equation of Motion](#equation-of-motion)
   - [First-Order Form](#first-order-form)
   - [Phase Space](#phase-space)
2. [Integrating the Equations](#integrating-the-equations)
   - [Explicit Euler](#explicit-euler)
   - [What Went Wrong?](#what-went-wrong)
   - [Semi-Implicit (Symplectic) Euler](#semi-implicit-symplectic-euler)
3. [Rotational Motion](#rotational-motion)
   - [The Pendulum Is a Rotational System](#pendulum-is-rotational)
   - [The Compound Pendulum](#compound-pendulum)
   - [Moment of Inertia](#moment-of-inertia)
   - [From 1D to 3D: the Inertia Tensor](#inertia-tensor-3d)
   - [Euler's Equations and the Dzhanibekov Effect](#euler-equations)
4. [General Rigid Body Equations of Motion](#general-rigid-body-eom)
   - [Position, Orientation, and Velocity](#position-orientation-velocity)
   - [Forces, Torques, and the Separation of Dynamics](#forces-torques)
   - [Momenta as State Variables](#momenta-state-variables)
   - [The State Vector and the Body-Space Trick](#state-vector)
   - [Why Quaternions?](#why-quaternions)
   - [Application: Quadcopter Dynamics](#quadcopter-dynamics)
5. [Contact, Collisions, and Friction](#contact-collisions-friction)
   - [Contact Normal Forces](#contact-normal-forces)
   - [Friction](#friction)
   - [The Friction Cone](#the-friction-cone)
   - [Block on an Inclined Plane](#block-on-inclined-plane)
6. [Deeper: Symplecticity and Higher-Order Methods](#deeper-symplecticity)
   - [Why Symplectic Euler Works](#why-symplectic-euler-works)
   - [Runge-Kutta Methods](#runge-kutta-methods)
7. [Stability Analysis](#stability-analysis)
8. [Stiff Systems and Implicit Methods](#stiff-systems-and-implicit-methods)
9. [Summary](#summary)

In [ ]:
%matplotlib widget
import sys
from pathlib import Path

_lecture_dir = Path.cwd()
if str(_lecture_dir) not in sys.path:
    sys.path.insert(0, str(_lecture_dir))

import numpy as np

from lib.pendulum_viz import (
    show_pendulum_diagram,
    show_pendulum_phase_space,
    show_pendulum_explicit_euler,
    show_pendulum_symplectic_euler,
    show_pendulum_euler_vs_symplectic,
    show_pendulum_euler_vs_symplectic_interactive,
    show_pendulum_comparison,
    show_pendulum_interactive,
)

---
# 1. The Simple Pendulum <a id="the-simple-pendulum"></a>

A mass $m$ hangs from a rigid massless rod of length $L$, pivoting freely at the top. Gravity $g$ pulls it downward.

The single degree of freedom is the angle $\theta$ measured from the downward vertical.

This is the simplest mechanical system with nontrivial dynamics — and it will teach us everything we need about numerical integration.

In [ ]:
show_pendulum_diagram(L=1.0, theta_deg=30.0)

## Equation of Motion <a id="equation-of-motion"></a>

Newton's second law in the **tangential** direction (along the arc):

$$m L \ddot{\theta} = -mg\sin\theta$$

Dividing by $mL$:

$$\boxed{\ddot{\theta} = -\frac{g}{L}\sin\theta}$$

This is a **second-order ODE** — $\ddot{\theta}$ depends on $\theta$.

Note: for small angles $\sin\theta \approx \theta$, and we get the linear harmonic oscillator $\ddot\theta = -(g/L)\theta$ with exact solution $\theta(t) = \theta_0 \cos(\sqrt{g/L}\,t)$. But we want to handle the **full nonlinear** equation.

## First-Order Form <a id="first-order-form"></a>

Numerical integrators work with **first-order** ODEs $\dot{y} = f(t, y)$. We convert by introducing an auxiliary variable:

$$\omega \equiv \dot{\theta} \qquad \text{(angular velocity)}$$

Now the single second-order equation becomes a **system of two first-order equations:**

$$\dot{\theta} = \omega$$
$$\dot{\omega} = -\frac{g}{L}\sin\theta$$

In vector form with state $y = (\theta, \omega)$:

$$\dot{y} = f(y) = \begin{pmatrix} \omega \\ -(g/L)\sin\theta \end{pmatrix}$$

This is the **general form** that every dynamical system can be put into — and the form that numerical integrators consume.

## Phase Space <a id="phase-space"></a>

The state $y = (\theta, \omega)$ lives in a 2D **phase space**. A point in this space fully determines the future evolution of the system.

The pendulum conserves energy:

$$E = \underbrace{\frac{1}{2}\omega^2}_{\text{kinetic}} \underbrace{- \frac{g}{L}\cos\theta}_{\text{potential}} = \text{const}$$

Since $E$ is conserved, trajectories in phase space follow **level curves** of $E(\theta, \omega)$.

Three qualitative regimes:
- **Libration** (small $E$): closed orbits around $\theta = 0$ — the pendulum swings back and forth
- **Separatrix** ($E = g/L$): the trajectory that just barely reaches the top ($\theta = \pm\pi$)
- **Rotation** (large $E$): the pendulum goes over the top — open trajectories

In [ ]:
show_pendulum_phase_space(g_over_L=9.81)

---
# 2. Integrating the Equations <a id="integrating-the-equations"></a>

We have $\dot{y} = f(y)$ with a known initial condition $y_0 = (\theta_0, 0)$. We want to compute $y(t)$ for $t > 0$.

For the nonlinear pendulum there is no closed-form solution (it involves elliptic integrals). We must integrate **numerically**.

Choose a time step $h$ and compute $y_1, y_2, \ldots$ on a grid $t_n = n h$. The question: **how?**

## Explicit (Forward) Euler <a id="explicit-euler"></a>

The simplest possible scheme. Taylor-expand and drop higher-order terms:

$$y(t + h) = y(t) + h\,\dot{y}(t) + O(h^2) \approx y(t) + h\,f(y(t))$$

This gives the **explicit Euler** update:

$$\boxed{y_{n+1} = y_n + h \cdot f(y_n)}$$

For our pendulum:
$$\theta_{n+1} = \theta_n + h \cdot \omega_n$$
$$\omega_{n+1} = \omega_n - h \cdot \frac{g}{L}\sin\theta_n$$

Let's try it.

In [ ]:
show_pendulum_explicit_euler(theta0_deg=30, h=0.05, T=100)

## What Went Wrong? <a id="what-went-wrong"></a>

The trajectory spirals **outward** in phase space. Energy grows monotonically. The pendulum swings wider and wider — physically nonsensical.

This is not a bug. It's a **fundamental property** of explicit Euler applied to oscillatory systems.

To see why, consider the linearized case (small $\theta$, so $\sin\theta \approx \theta$). The ODE becomes $\dot{y} = Ay$ with

$$A = \begin{pmatrix} 0 & 1 \\ -g/L & 0 \end{pmatrix}$$

The eigenvalues are $\lambda = \pm i\sqrt{g/L}$ — purely imaginary (oscillation, no growth or decay).

One Euler step multiplies the state by the **amplification factor** $g(h\lambda) = 1 + h\lambda$. Its magnitude:

$$|1 + ih\omega_0| = \sqrt{1 + h^2\omega_0^2} > 1 \quad \text{always!}$$

Every single step **amplifies** the solution. After $N$ steps the error grows as $(1 + h^2\omega_0^2)^{N/2}$ — exponential blowup.

We need a method that doesn't add energy at every step.

## Semi-Implicit (Symplectic) Euler <a id="semi-implicit-symplectic-euler"></a>

A tiny modification: **use the new velocity in the position update:**

$$\boxed{\omega_{n+1} = \omega_n - h \cdot \frac{g}{L}\sin\theta_n \qquad \theta_{n+1} = \theta_n + h \cdot \omega_{n+1}}$$

Compare to explicit Euler:

| | Explicit Euler | Symplectic Euler |
|---|---|---|
| velocity update | $\omega_{n+1} = \omega_n + h\,a(\theta_n)$ | $\omega_{n+1} = \omega_n + h\,a(\theta_n)$ |
| position update | $\theta_{n+1} = \theta_n + h \cdot \omega_n$ | $\theta_{n+1} = \theta_n + h \cdot \omega_{\mathbf{n+1}}$ |

The velocity update is identical. The only difference: the position update uses the **already-computed new velocity** instead of the old one.

Let's see what this changes.

In [ ]:
show_pendulum_symplectic_euler(theta0_deg=30, h=0.05, T=20)

The phase portrait stays on a **closed curve**. Energy oscillates slightly but doesn't drift. The trajectory tracks the reference solution.

One changed line of code — from $\omega_n$ to $\omega_{n+1}$ in the position update — makes the difference between a useless and a useful integrator.

Let's see both methods side-by-side.

In [ ]:
show_pendulum_euler_vs_symplectic(theta0_deg=30, h=0.05, T=20)

In [ ]:
show_pendulum_euler_vs_symplectic_interactive(theta0_deg=30, T=20)

---
# 3. Rotational Motion <a id="rotational-motion"></a>

The pendulum is already a rotational system — the mass rotates about the pivot. We just didn't use rotational language. Let's make it explicit. This will introduce **moment of inertia**, **torque**, and **angular acceleration** — the building blocks of rigid body dynamics.

## The Pendulum Is a Rotational System <a id="pendulum-is-rotational"></a>

The mass traces a circular arc around the pivot — it's rotating. Let's rewrite Newton's second law in rotational terms.

**Torque** (rotational analog of force): force $\times$ lever arm.

$$\tau = -mg \cdot L \cdot \sin\theta$$

**Moment of inertia** (rotational analog of mass): how hard it is to spin the object.

$$I = mL^2 \qquad \text{(point mass at distance } L \text{ from pivot)}$$

**Angular acceleration:**

$$\alpha = \ddot\theta$$

**Newton's second law for rotation:**

$$\boxed{\tau = I\alpha} \qquad \Longrightarrow \qquad mL^2 \ddot\theta = -mgL\sin\theta \qquad \Longrightarrow \qquad \ddot\theta = -\frac{g}{L}\sin\theta$$

Exactly the same equation as before — but now we see it comes from $\tau = I\alpha$.

This is the rotational analog of $F = ma$:

| Translational | Rotational |
|---------------|------------|
| Force $F$ | Torque $\tau$ |
| Mass $m$ | Moment of inertia $I$ |
| Acceleration $a$ | Angular acceleration $\alpha = \ddot\theta$ |
| Velocity $v$ | Angular velocity $\omega = \dot\theta$ |
| Momentum $p = mv$ | Angular momentum $L = I\omega$ |
| $F = ma$ | $\tau = I\alpha$ |

## The Compound Pendulum <a id="compound-pendulum"></a>

What if we replace the point mass with a **rigid rod** of length $L$ and mass $M$, pivoting at one end?

Now the mass is distributed along the rod, not concentrated at the tip. This changes two things:

1. **Moment of inertia:** $I = \frac{1}{3}ML^2$ (not $ML^2$)
2. **Torque:** gravity acts at the center of mass, which is at $L/2$ from the pivot: $\tau = -Mg\frac{L}{2}\sin\theta$

The equation of motion becomes:

$$\frac{1}{3}ML^2\,\ddot\theta = -Mg\frac{L}{2}\sin\theta \qquad \Longrightarrow \qquad \ddot\theta = -\frac{3g}{2L}\sin\theta$$

Same form, **different effective frequency**: $\omega_0^2 = 3g/2L$ instead of $g/L$. The compound pendulum oscillates faster! The mass distribution — captured entirely by $I$ — determines the dynamics.

In [ ]:
from lib.rotation_intro_viz import show_compound_pendulum_diagram, show_simple_vs_compound_phase_space

show_compound_pendulum_diagram()

In [ ]:
show_simple_vs_compound_phase_space()

## Moment of Inertia <a id="moment-of-inertia"></a>

In general, for a continuous body rotating about an axis:

$$I = \int r_\perp^2 \, dm$$

where $r_\perp$ is the perpendicular distance from the rotation axis to each mass element $dm$.

| Shape | Axis | $I$ |
|-------|------|-----|
| Point mass $m$ at distance $R$ | Through pivot | $mR^2$ |
| Thin rod, mass $M$, length $L$ | Through center | $\frac{1}{12}ML^2$ |
| Thin rod, mass $M$, length $L$ | Through end | $\frac{1}{3}ML^2$ |
| Solid sphere, mass $M$, radius $R$ | Through center | $\frac{2}{5}MR^2$ |
| Solid cylinder, mass $M$, radius $R$ | Symmetry axis | $\frac{1}{2}MR^2$ |
| Solid box, sides $a, b, c$ | Through center, $\perp$ to $a$ face | $\frac{M}{12}(b^2 + c^2)$ |

**Parallel axis theorem:** if $I_{\text{cm}}$ is the moment of inertia about an axis through the center of mass, then about a parallel axis at distance $d$:

$$I = I_{\text{cm}} + Md^2$$

This is how we got the rod's $I$ about the end: $\frac{1}{12}ML^2 + M(L/2)^2 = \frac{1}{3}ML^2$.

> *"If a book and a banana have the same inertia tensor, then if they are thrown in the same way the subsequent motion will be the same, however complicated that motion is."* — Sussman & Wisdom, *Structure and Interpretation of Classical Mechanics*

## From 1D to 3D: the Inertia Tensor <a id="inertia-tensor-3d"></a>

So far we've considered rotation about a single fixed axis — one angle $\theta$, one scalar $I$.

In 3D, a body can rotate about **any** axis. The moment of inertia becomes a $3\times3$ symmetric matrix — the **inertia tensor**:

$$\mathbf{I} = \sum_\alpha m_\alpha \left( |\xi_\alpha|^2 \mathbf{1} - \xi_\alpha \otimes \xi_\alpha \right)$$

where $\xi_\alpha$ is the position of particle $\alpha$ relative to the center of mass.

Since $\mathbf{I}$ is symmetric, it can be **diagonalized**. The eigenvectors are the **principal axes**; the eigenvalues $A \leq B \leq C$ are the **principal moments of inertia**.

In the principal-axis frame:

$$\mathbf{I} = \text{diag}(A, B, C)$$

The rotational kinetic energy takes the elegant form:

$$T_{\text{rot}} = \frac{1}{2}\left(A\omega_a^2 + B\omega_b^2 + C\omega_c^2\right)$$

and the angular momentum components are $L_a = A\omega_a$, $L_b = B\omega_b$, $L_c = C\omega_c$.

## Euler's Equations and the Dzhanibekov Effect <a id="euler-equations"></a>

For a rigid body spinning freely in space (**no external torques**), the equations of motion in the body frame are **Euler's rotation equations:**

$$A\,\dot\omega_a = (B - C)\,\omega_b\,\omega_c$$
$$B\,\dot\omega_b = (C - A)\,\omega_c\,\omega_a$$
$$C\,\dot\omega_c = (A - B)\,\omega_a\,\omega_b$$

Two conserved quantities:
- **Kinetic energy:** $T = \frac{1}{2}(A\omega_a^2 + B\omega_b^2 + C\omega_c^2)$
- **Angular momentum magnitude:** $|L|^2 = A^2\omega_a^2 + B^2\omega_b^2 + C^2\omega_c^2$

The cross-product coupling terms $(B-C)\omega_b\omega_c$ etc. have no analog in translational motion — they are the source of all the surprises in rotational dynamics.

**Stability of rotation about principal axes** (with $A < B < C$):
- Rotation about the axis of **smallest** $I$ ($A$): **stable**
- Rotation about the axis of **largest** $I$ ($C$): **stable**
- Rotation about the **intermediate** axis ($B$): **UNSTABLE**

This is the **tennis racket theorem** (or **Dzhanibekov effect**): a book thrown spinning about its intermediate axis tumbles chaotically.

<video width="600" autoplay muted loop>
    <source src="assets/dzhanibekov.mp4" type="video/mp4">
</video>

In [ ]:
from lib.rigid_body_rotation_viz import (
    show_euler_equations_stability,
    show_dzhanibekov_effect,
    show_dzhanibekov_interactive,
)

show_euler_equations_stability(I1=1.0, I2=2.0, I3=3.0)

In [ ]:
show_dzhanibekov_effect(I1=1.0, I2=2.0, I3=3.0, perturbation=0.1, T=20.0)

In [ ]:
show_dzhanibekov_interactive()

---
# 4. General Rigid Body Equations of Motion <a id="general-rigid-body-eom"></a>

Euler's equations describe the angular velocity of a freely spinning body. But a real rigid body also **translates** — it flies through the air, slides across a surface, gets pushed by actuators. We need the **full** equations of motion that couple translation and rotation, with arbitrary external forces and torques.

This is the foundation of every physics engine used in robotics (MuJoCo, Bullet, PhysX, Drake). The derivation follows Baraff's classic treatment and standard analytical mechanics.

## Position, Orientation, and Velocity <a id="position-orientation-velocity"></a>

### Body space and world space

A rigid body has a fixed shape. We describe it in a **body-space** coordinate frame where the center of mass sits at the origin. To place it into the world, we apply a rotation $R(t)$ and a translation $x(t)$:

$$p(t) = R(t)\,p_0 + x(t)$$

where $p_0$ is any point on the body in body space and $p(t)$ is its world-space position at time $t$.

- $x(t) \in \mathbb{R}^3$ — position of the center of mass (3 DOF)
- $R(t) \in SO(3)$ — a $3\times3$ rotation matrix (3 DOF)

### Linear velocity

$$v(t) = \dot{x}(t) = \frac{dx}{dt}$$

This is the velocity of the center of mass — the translational part.

### Angular velocity

The rotation matrix changes as the body spins. The relationship is:

$$\dot{R}(t) = [\omega(t)]_\times \, R(t)$$

where $[\omega]_\times$ is the **skew-symmetric matrix** formed from $\omega$:

$$[\omega]_\times = \begin{pmatrix} 0 & -\omega_z & \omega_y \\ \omega_z & 0 & -\omega_x \\ -\omega_y & \omega_x & 0 \end{pmatrix}$$

so that $[\omega]_\times\, a = \omega \times a$ for any vector $a$.

### Velocity of any point on the body

Differentiating $p(t) = R(t)\,p_0 + x(t)$ gives:

$$\dot{p}(t) = v(t) + \omega(t) \times \bigl(p(t) - x(t)\bigr)$$

Every point's velocity decomposes into a **linear** part $v$ (same for all points) and an **angular** part $\omega \times r'$ (depends on position relative to CM).

## Forces, Torques, and the Separation of Dynamics <a id="forces-torques"></a>

Imagine the body as a collection of particles with masses $m_i$ at positions $r_i(t)$.

**Total force** — the sum of all external forces:

$$F(t) = \sum_i F_i(t)$$

**Torque** about the center of mass — force $\times$ lever arm:

$$\tau(t) = \sum_i \bigl(r_i(t) - x(t)\bigr) \times F_i(t)$$

A key insight: because we work in a **center-of-mass frame** ($\sum m_i (r_i - x) = 0$), the translational and rotational dynamics **decouple**:

- $F$ determines how the center of mass accelerates — it knows nothing about rotation.
- $\tau$ determines how the body spins — it knows nothing about translation.

A uniform gravity field $g$ exerts force $Mg$ at the center of mass and **zero torque** about it. Gravity alone cannot spin a free-flying body.

## Momenta as State Variables <a id="momenta-state-variables"></a>

### Linear momentum

$$P(t) = M\,v(t)$$

Newton's second law: $\quad \boxed{\dot{P}(t) = F(t)}$

### Angular momentum

$$L(t) = I(t)\,\omega(t)$$

The rotational analog: $\quad \boxed{\dot{L}(t) = \tau(t)}$

### Why momenta instead of velocities?

The time derivatives of momenta are beautifully simple — just the applied force/torque. In contrast, $\dot\omega$ involves the messy cross-product terms from Euler's equations:

$$I\,\dot\omega = \tau - \omega \times (I\omega)$$

We already saw that angular momentum $L$ is conserved when $\tau = 0$, but angular velocity $\omega$ is **not** — the Dzhanibekov effect proves this. Momenta are the natural state variables.

This is why physics engines (MuJoCo, Bullet) store $(P, L)$, not $(v, \omega)$.

## The State Vector and the Body-Space Trick <a id="state-vector"></a>

### The state vector

The complete state of a rigid body is described by **13 numbers**:

$$\mathbf{Y}(t) = \begin{pmatrix} x(t) \\ q(t) \\ P(t) \\ L(t) \end{pmatrix} \in \mathbb{R}^{13} \qquad \begin{array}{l} x \in \mathbb{R}^3 \text{ — position of CM} \\ q \in \mathbb{R}^4 \text{ — orientation quaternion} \\ P \in \mathbb{R}^3 \text{ — linear momentum} \\ L \in \mathbb{R}^3 \text{ — angular momentum} \end{array}$$

### The derivative

$$\boxed{\frac{d}{dt}\mathbf{Y}(t) = \begin{pmatrix} P(t)/M \\ \frac{1}{2}\,[0,\, \omega(t)] \otimes q(t) \\ F(t) \\ \tau(t) \end{pmatrix}}$$

where $[0, \omega]$ is the world-frame angular velocity promoted to a pure quaternion and $\otimes$ is the quaternion product.

This is an ODE of the form $\dot{\mathbf{Y}} = f(t, \mathbf{Y})$ — exactly what our integrators (Euler, RK4, ...) need.

### Computing $\omega$ from $L$: the body-space trick

We need $\omega$ to compute $\dot{q}$. The inertia tensor $I(t)$ changes with orientation:

$$I(t) = R(t)\,I_{\text{body}}\,R(t)^T$$

But $I_{\text{body}}$ is a **constant** matrix computed once from the body's geometry. Then:

$$\omega = I(t)^{-1}\,L = R\,I_{\text{body}}^{-1}\,R^T\,L$$

No integrals at runtime — just a matrix rotation.

**Example:** a box with sides $a \times b \times c$ and mass $M$:

$$I_{\text{body}} = \frac{M}{12}\begin{pmatrix} b^2+c^2 & 0 & 0 \\ 0 & a^2+c^2 & 0 \\ 0 & 0 & a^2+b^2 \end{pmatrix}$$

### The simulation recipe

Given the current state $\mathbf{Y}(t)$:

1. **Unpack** $x, q, P, L$ from the state vector
2. **Compute** $R$ from $q$, then $I^{-1}(t) = R\,I_{\text{body}}^{-1}\,R^T$
3. **Recover** $v = P/M$ and $\omega = I^{-1}(t)\,L$
4. **Evaluate** external forces $F(t)$ and torques $\tau(t)$
5. **Assemble** $\dot{\mathbf{Y}} = (v,\; \tfrac{1}{2}[0,\omega] \otimes q,\; F,\; \tau)$
6. **Feed** to the integrator (RK4, symplectic Euler, ...)

## Why Quaternions? <a id="why-quaternions"></a>

We represent orientation with a **unit quaternion** $q = [w, x, y, z]$ rather than a rotation matrix $R$. Why?

| | Rotation matrix $R$ | Quaternion $q$ |
|---|---|---|
| Parameters | 9 (for 3 DOF) | 4 (for 3 DOF) |
| Constraint | $R^T R = I$, $\det R = 1$ (6 constraints) | $\|q\| = 1$ (1 constraint) |
| Drift under integration | Orthogonality loss → **skewing** | Magnitude drift → just **renormalize** |
| Derivative | $\dot{R} = [\omega]_\times R$ (9 equations) | $\dot{q} = \frac{1}{2}[0, \omega] \otimes q$ (4 equations) |

Quaternion drift is corrected by a single division: $q \leftarrow q / \|q\|$.

Matrix drift requires a Gram-Schmidt re-orthogonalization (expensive and numerically fragile).

This is why **every modern physics engine** uses quaternions internally.

### Simulation: tumbling box

Let's put the recipe to work. A box ($1 \times 2 \times 3$ m, $M = 5$ kg) is tossed upward with initial spin $\omega_0 = (0.5, 4.0, 0.2)$ rad/s. Gravity is the only external force, acting at the center of mass — so the torque about CM is zero and angular momentum $L$ should be conserved exactly.

We integrate the 13-dimensional state vector with RK4.

In [ ]:
from lib.rigid_body_sim_viz import show_tumbling_box

show_tumbling_box()

## Application: Quadcopter Dynamics <a id="quadcopter-dynamics"></a>

A quadcopter is an ideal testbed for the general rigid body equations we just derived: it has 6 DOF (3 translational + 3 rotational), only 4 control inputs (rotor speeds), a diagonal inertia tensor, and the forces/torques map directly to our framework.

### How 4 rotors control 6 DOF

A quadcopter has four rotors at the corners of a cross, spinning in alternating directions (1 & 3 clockwise, 2 & 4 counterclockwise). Each rotor produces thrust proportional to the **square** of its angular speed:

$$T_i = k\,\omega_i^2$$

The total **thrust** (in the body frame, along $z_B$) is:

$$T_B = \begin{pmatrix} 0 \\ 0 \\ k\sum_{i=1}^{4}\omega_i^2 \end{pmatrix}$$

**Roll** torque (differential thrust between rotors 1 and 3):

$$\tau_\phi = Lk\left(\omega_1^2 - \omega_3^2\right)$$

**Pitch** torque (differential thrust between rotors 2 and 4):

$$\tau_\theta = Lk\left(\omega_2^2 - \omega_4^2\right)$$

**Yaw** torque (from rotor drag — CW vs CCW rotors produce opposite reaction torques):

$$\tau_\psi = b\left(\omega_1^2 - \omega_2^2 + \omega_3^2 - \omega_4^2\right)$$

where $L$ is the arm length, $k$ is the thrust coefficient, and $b$ is the drag torque coefficient.

### Equations of motion

**Linear** (in the inertial frame):

$$m\ddot{x} = \begin{pmatrix}0\\0\\-mg\end{pmatrix} + R\,T_B - k_d\,\dot{x}$$

where $R(\phi, \theta, \psi)$ is the rotation matrix that maps body frame to inertial frame, and $k_d$ is a linear drag coefficient.

**Rotational** (in the body frame — exactly Euler's equations with specific torques):

$$I\dot\omega = \tau - \omega \times (I\omega)$$

With a diagonal inertia tensor $I = \text{diag}(I_{xx}, I_{yy}, I_{zz})$, this expands to:

$$\dot\omega_x = \frac{\tau_\phi + (I_{yy} - I_{zz})\,\omega_y\,\omega_z}{I_{xx}}, \quad \dot\omega_y = \frac{\tau_\theta + (I_{zz} - I_{xx})\,\omega_x\,\omega_z}{I_{yy}}, \quad \dot\omega_z = \frac{\tau_\psi + (I_{xx} - I_{yy})\,\omega_x\,\omega_y}{I_{zz}}$$

This is a 12-dimensional ODE — exactly the kind of system our integrators can handle.

In [ ]:
from lib.quadcopter_viz import show_quadcopter_diagram

show_quadcopter_diagram()

### What happens when two rotors fail?

Suppose the drone is hovering with all four rotors at the speed needed to counteract gravity: $\omega_{\text{hover}}^2 = mg/(4k)$. At $t = 0$ rotors 3 and 4 suddenly stop. Rotors 1 and 2 keep spinning at the same speed. No controller, no feedback — just the raw equations of motion with **constant inputs**.

The consequences follow directly from our equations:

1. **Reduced thrust**: only half the thrust needed for hover $\Rightarrow$ the drone starts **falling**.
2. **Roll torque**: rotor 1 pushes up on the left arm, but rotor 3 (its counterpart on the right) is dead $\Rightarrow$ net $\tau_\phi = Lk\omega_1^2 \neq 0$ $\Rightarrow$ the drone **rolls**.
3. **Pitch torque**: similarly, rotor 2 is alive but rotor 4 is dead $\Rightarrow$ net $\tau_\theta = Lk\omega_2^2 \neq 0$ $\Rightarrow$ the drone **pitches**.
4. **Yaw torque**: rotor 1 (CW) and rotor 2 (CCW) exert opposite yaw torques that largely cancel, but not perfectly $\Rightarrow$ slow **yaw**.
5. **Translation from tilt**: as the body tilts, the thrust vector (always along body $z$) gets a horizontal component $\Rightarrow$ the drone **drifts sideways** while falling.

All of this is what the numerical integrator will produce when we feed it the 12D ODE with constant $\gamma = [\omega^2, \omega^2, 0, 0]$.

We start with a level drone at 10 m altitude, zero velocity, and integrate with RK4. The simulation stops when the drone hits the ground.

In [ ]:
from lib.quadcopter_viz import show_quadcopter_open_loop

show_quadcopter_open_loop(active_rotors=(1, 2), T=2.5)

### Takeaways

- **No mystery**: we wrote down the forces and torques, plugged them into $\dot Y = f(Y)$, and let the integrator compute the trajectory. That's all a physics engine does.
- **Rotation–translation coupling**: even though the rotor forces point along the body $z$-axis, the tilting body redirects thrust horizontally. The drone doesn't just fall — it **curves** away. This coupling is inherent in the rotation matrix $R$ that appears in the linear EOM.
- **Euler cross-terms** $\omega \times (I\omega)$ produce yaw even though neither active rotor directly creates a large yaw torque — it emerges from the coupling between roll and pitch rates.
- This exact ODE structure — with specific forces and torques plugged into the general rigid body equations — is what MuJoCo, Bullet, and Drake solve at every timestep. The only extras are more complex force models (contact, friction, actuator dynamics).

---
# 5. Contact, Collisions, and Friction <a id="contact-collisions-friction"></a>

Until now, our bodies fly freely through space. Real robots **touch things**. When two objects collide or rest against each other, we need **contact forces** to prevent interpenetration. This section covers the two main approaches and how friction fits in.

## Contact Normal Forces

The problem: objects must not overlap, but our ODE knows nothing about geometry. Two standard solutions:

### Penalty method

Treat the contact surface as a stiff spring. If an object penetrates by depth $\delta$:

$$F_n = k\,\delta + c\,\dot\delta$$

where $k$ is contact stiffness and $c$ is damping. Simple and widely used — MuJoCo uses a variant. **Downside**: makes the system **stiff**, requiring small timesteps or implicit integration.

### Impulse-based method

Detect collision events and instantaneously change velocity:

$$v^+ = -e\,v^-$$

where $e \in [0, 1]$ is the **coefficient of restitution**:
- $e = 1$: perfectly elastic (total kinetic energy conserved)
- $e = 0$: perfectly inelastic (objects stick together)

Both methods are used in practice; many engines combine them (penalty forces for resting contact, impulses for fast collisions).

In [ ]:
from lib.bouncing_ball_viz import show_bouncing_ball_comparison, show_restitution_comparison

show_bouncing_ball_comparison()

In [ ]:
show_restitution_comparison()

## Friction

Once objects are in contact, tangential forces resist sliding.

**Coulomb friction**: the tangential force is bounded by

$$|F_t| \leq \mu\, F_n$$

Maximum static friction $= \mu_s F_n$; kinetic friction $= \mu_k F_n$, directed opposite to the velocity.

**The problem for simulation:** the Coulomb model is **discontinuous at $v = 0$**. Stiction (zero velocity with nonzero applied force) can't be represented as a smooth ODE — the force depends on a condition ($v = 0$ or $v \neq 0$), not a smooth function of state.

**Regularized models** replace the discontinuity with a smooth approximation:

- **Regularized:** $\quad F_t = -\mu\, F_n \cdot \dfrac{v}{\max(|v|,\, \varepsilon)}$

- **Tanh:** $\quad F_t = -\mu\, F_n \cdot \tanh\!\left(\dfrac{v}{\varepsilon}\right)$

These trade physical accuracy near $v = 0$ for numerical tractability.

## The Friction Cone

In 3D, the Coulomb constraint $|F_t| \leq \mu\, F_n$ defines a **cone** in force space $(F_{t1}, F_{t2}, F_n)$.

Any valid contact force must lie **inside** this cone. The apex is at the origin (zero force) and the cone opens in the $F_n$ direction. A force vector that exits the cone means the contact is slipping — friction cannot hold it.

For optimization-based solvers (LCP, QP), the cone is often approximated by a **friction pyramid** — a polyhedral inner approximation with $k$ flat facets. This turns the nonlinear cone constraint into a set of linear inequalities, which is much easier to solve.

In [ ]:
from lib.friction_block_viz import show_friction_cone

show_friction_cone(mu=0.5)

## Block on an Inclined Plane

The classic example: a block of mass $m$ on a plane tilted at angle $\alpha$.

Forces:
- Gravity component along the slope: $mg\sin\alpha$
- Normal force: $F_n = mg\cos\alpha$

The block is **static** if $\tan\alpha \leq \mu_s$ (friction can balance gravity). It **slides** if $\tan\alpha > \mu_s$.

Let's compare the three friction models (discontinuous Coulomb, regularized, tanh) in simulation.

In [ ]:
from lib.friction_block_viz import show_friction_comparison, show_static_vs_kinetic

show_friction_comparison(alpha_deg=35.0, mu=0.5)

In [ ]:
show_static_vs_kinetic(mu=0.5)

### Takeaways

- **Penalty forces** and **impulse methods** are two complementary approaches to contact: penalty forces integrate naturally into the ODE, while impulses handle discrete collision events.
- Friction is inherently **non-smooth** — regularization is the standard workaround in simulation.
- The **friction cone** is a fundamental geometric object in robot manipulation (grasp planning, force closure analysis).
- **What we didn't cover:** holonomic constraints (joints, hinges), which are handled via Lagrange multipliers or constraint stabilization (Baumgarte). These are critical for articulated robots but deserve their own lecture.

---
# 6. Deeper: Symplecticity and Higher-Order Methods <a id="deeper-symplecticity"></a>

## Why Symplectic Euler Works <a id="why-symplectic-euler-works"></a>

A mapping $(\theta_n, \omega_n) \mapsto (\theta_{n+1}, \omega_{n+1})$ is **symplectic** if it preserves the symplectic 2-form $d\omega \wedge d\theta$ (i.e., phase-space area in 1D).

**Key theorem (backward error analysis):**  
A symplectic integrator exactly solves a **nearby Hamiltonian** $\tilde{H} = H + O(h)$.  
Therefore energy oscillates around the true value with **bounded error** — it never drifts.

### This Is What Real Simulators Use

| Engine | Integrator |
|--------|------------|
| **MuJoCo** | Semi-implicit Euler (+ Runge-Kutta option) |
| **Bullet Physics** | Symplectic Euler |
| **PhysX** | Symplectic Euler with substeps |
| **Isaac Gym / Isaac Sim** | Semi-implicit |

Why? It's only $O(h)$ accurate, but it's **stable for Hamiltonian systems** and costs just **one force evaluation per step**.

## Runge-Kutta Methods <a id="runge-kutta-methods"></a>

**Idea:** evaluate $f$ at multiple points within the step to achieve higher-order accuracy.

### RK2 (Midpoint Method)

$$k_1 = f(t_n, y_n)$$
$$k_2 = f\!\left(t_n + \tfrac{h}{2}, \; y_n + \tfrac{h}{2} k_1\right)$$
$$y_{n+1} = y_n + h \cdot k_2$$

Global error: $O(h^2)$. Cost: 2 function evaluations per step.

### RK4 (Classic 4th-order)

$$k_1 = f(t_n, y_n)$$
$$k_2 = f\!\left(t_n + \tfrac{h}{2}, \; y_n + \tfrac{h}{2} k_1\right)$$
$$k_3 = f\!\left(t_n + \tfrac{h}{2}, \; y_n + \tfrac{h}{2} k_2\right)$$
$$k_4 = f(t_n + h, \; y_n + h k_3)$$
$$y_{n+1} = y_n + \frac{h}{6}(k_1 + 2k_2 + 2k_3 + k_4)$$

Global error: $O(h^4)$. Cost: 4 function evaluations per step.

### Butcher Tableau

Compact notation for any RK method:

$$\begin{array}{c|cccc}
0 & & & & \\
1/2 & 1/2 & & & \\
1/2 & 0 & 1/2 & & \\
1 & 0 & 0 & 1 & \\
\hline
& 1/6 & 1/3 & 1/3 & 1/6
\end{array}$$

### Adaptive Step Size: RK45 (Dormand-Prince)

Compute both 4th-order and 5th-order solutions simultaneously; the difference estimates the local error.

- If error $> \text{tol}$: shrink $h$
- If error $\ll \text{tol}$: grow $h$

This is the default method in `scipy.integrate.solve_ivp`.

### When to Use

- **RK4/RK45:** offline trajectory optimization, high-precision orbit calculations
- **NOT** real-time simulation (too many $f$-evaluations per step when $h$ is constrained by real-time requirements)

In [ ]:
show_pendulum_comparison(theta0_deg=170.0, T=20.0, h=0.01)

In [ ]:
show_pendulum_interactive(theta0_deg=170.0, T=20.0)

---
# 7. Stability Analysis <a id="stability-analysis"></a>

In [ ]:
from lib.stability_viz import (
    show_stability_regions,
    show_stability_regions_overlay,
    show_stability_with_examples,
)

### The Test Equation

$$\dot{y} = \lambda y, \qquad \lambda \in \mathbb{C}$$

Any linear system $\dot{y} = Ay$ can be diagonalized into independent test equations. The eigenvalues $\lambda_i$ of $A$ determine the behavior.

### Amplification Factor

After one step: $y_{n+1} = g(h\lambda) \cdot y_n$. For stability: $|g(h\lambda)| \leq 1$.

| Method | $g(z)$ where $z = h\lambda$ | Stability region |
|--------|------|-------------------|
| Explicit Euler | $1 + z$ | Disk of radius 1 centered at $-1$ |
| RK2 | $1 + z + z^2/2$ | Larger region |
| RK4 | $1 + z + z^2/2 + z^3/6 + z^4/24$ | Even larger |
| Implicit Euler | $1/(1-z)$ | Entire left half-plane (**A-stable**) |

### Key Insight

**Accuracy and stability are different things.**

A method can be accurate (small truncation error) but unstable (errors grow exponentially).  
In real-time simulation with a fixed timestep, **stability matters more than order of accuracy**.

For a pure oscillator ($\lambda = i\omega$):
- Explicit Euler: $|g| = \sqrt{1 + h^2\omega^2} > 1$ — always unstable
- RK4: $|g| \approx 1$ for small $h\omega$, but $> 1$ eventually
- Implicit Euler: $|g| = 1/\sqrt{1 + h^2\omega^2} < 1$ — always stable, but **dissipative**

In [ ]:
show_stability_regions()

In [ ]:
show_stability_regions_overlay()

In [ ]:
show_stability_with_examples()

---
# 8. Stiff Systems and Implicit Methods <a id="stiff-systems-and-implicit-methods"></a>

In [ ]:
from lib.stiff_system_viz import (
    show_stiff_system_comparison,
    show_stiff_explicit_euler_h_sweep,
    show_van_der_pol_stiff,
    show_stiff_interactive,
)

### What Is Stiffness?

A system is **stiff** when the Jacobian $\partial f / \partial y$ has eigenvalues with very different magnitudes.

**Example:** a robot arm with a very stiff joint ($k = 10^4$) and a flexible payload ($k = 1$).

The stiff mode requires $h < \frac{2}{\sqrt{k_{\text{stiff}}/m}}$ for explicit stability, even though the motion we care about (the payload) is slow.

### Implicit (Backward) Euler

$$\boxed{y_{n+1} = y_n + h \cdot f(t_{n+1}, y_{n+1})}$$

Note: $f$ is evaluated at the **unknown** $y_{n+1}$. This is a nonlinear equation that must be solved iteratively.

**Newton's method:**
$$y^{(k+1)} = y^{(k)} - \left(I - h\frac{\partial f}{\partial y}\right)^{-1} \left(y^{(k)} - y_n - h\,f(t_{n+1}, y^{(k)})\right)$$

**Cost per step:**
- Need the Jacobian $\partial f / \partial y$ (analytic or finite-difference)
- Need to solve a linear system at each Newton iteration
- Expensive per step, but can take **much larger** $h$

### Stability Properties

- **A-stable:** $|g(z)| \leq 1$ for all $\text{Re}(z) \leq 0$ (the entire left half-plane)
- **L-stable:** additionally $|g(z)| \to 0$ as $\text{Re}(z) \to -\infty$ (damps fast modes completely)

Implicit Euler is both A-stable and L-stable.

### Energy Behavior

| Method | Energy behavior |
|--------|----------------|
| Explicit Euler | Energy **grows** (adds energy) |
| Symplectic Euler | Energy **oscillates** (bounded) |
| RK4 | Energy drifts **slowly** |
| Implicit Euler | Energy **dissipates** (numerical damping) |

Some simulators (like MuJoCo) use implicit-Euler-like damping deliberately to stabilize contact dynamics.

### BDF Methods

**BDF2** (2-step backward differentiation formula):
$$y_{n+1} = \frac{4}{3}y_n - \frac{1}{3}y_{n-1} + \frac{2h}{3}f(t_{n+1}, y_{n+1})$$

Order 2, A-stable. Used by `scipy.integrate.solve_ivp(method='BDF')` and MATLAB's `ode15s`.

In [ ]:
show_stiff_system_comparison(k_stiff=1e4, k_soft=1.0, T=10.0, h=0.01)

In [ ]:
show_stiff_explicit_euler_h_sweep(k_stiff=1e4, k_soft=1.0, T=2.0)

In [ ]:
show_van_der_pol_stiff(mu=1000.0, T=3000.0)

In [ ]:
show_stiff_interactive()

---
# 9. Summary <a id="summary"></a>

## Numerical Methods

| Method | Order | Stability | Energy | Cost/step | Used in |
|--------|-------|-----------|--------|-----------|---------|
| Explicit Euler | 1 | Conditional | Grows | 1 f-eval | Teaching |
| **Symplectic Euler** | 1 | Conditional (symplectic) | **Oscillates** | 1 f-eval | **MuJoCo, Bullet, PhysX** |
| RK4 | 4 | Conditional (larger) | Slow drift | 4 f-evals | Offline sim |
| Implicit Euler | 1 | **A-stable** | Dissipates | Newton iters | Stiff contact |
| BDF2 | 2 | **A-stable** | Mild dissipation | Newton iters | scipy, MATLAB |

## Key Takeaways

1. Convert second-order equations to first-order by introducing velocity as an auxiliary variable
2. Trajectories in phase space follow level curves of conserved quantities (energy)
3. Explicit Euler **adds energy** at every step — fundamentally unsuitable for oscillatory systems
4. **Symplectic Euler** preserves phase-space structure — the workhorse of real-time robotics simulation
5. **Stability $\neq$ accuracy**: with fixed timestep, stability matters more
6. **Stiff systems** (stiff contacts, stiff joints) require implicit methods or very small timesteps

## References

- Hairer, Nørsett, Wanner, *Solving ODEs I* — Runge-Kutta and stability
- Hairer, Lubich, Wanner, *Geometric Numerical Integration* — symplectic methods
- MuJoCo documentation: mujoco.readthedocs.io
- Featherstone, *Rigid Body Dynamics Algorithms*